# Save or Let Go — Week 4 mini lab

This notebook has three passes through the same churn problem:

1. **Look first — inspect correlations and group differences**
2. **Build the baseline — fit logistic regression with one input, two inputs, and then the full feature set**
3. **Redo with regularization — repeat the full comparison with Ridge and Lasso**

Use **File → Save a copy in Drive** before editing. Stop at each **Discuss** prompt and compare your result with a partner. The later sections remain as a workspace for completing Exercise 02.


## Before you run anything

**Prediction question:** What is the probability that a customer will churn?

**Decision question:** Which customers should receive a retention contact, given limited capacity and campaign costs?

The model produces a probability. A threshold and the campaign assumptions turn that probability into an action.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42


## Load the data

The loader first looks for a local repository checkout. In Colab, it reads the same files directly from the public GitHub repository, so no authorization step should be required.


In [ ]:
LOCAL_DATA = Path("../data")
RAW_BASE = "https://raw.githubusercontent.com/lathrahul/modern-ai-ml-exercises/main/exercise-02-retention/data"

def read_course_csv(filename):
    local_file = LOCAL_DATA / filename
    source = local_file if local_file.exists() else f"{RAW_BASE}/{filename}"
    return pd.read_csv(source)

train = read_course_csv("train.csv")
test = read_course_csv("test.csv")
costs = read_course_csv("campaign_costs.csv")

print("train:", train.shape, "test:", test.shape)
train.head()


# Look first — relationships in the data

Correlation is a descriptive starting point. It tells us whether two numeric variables move together in this sample. It does **not** produce a churn probability, control for other variables, or establish causality.

Because `churned` is coded 0/1, its Pearson correlation with a numeric feature is also the point-biserial correlation.


In [ ]:
target = "churned"
numeric_for_corr = ["tenure_months", "monthly_charges", "total_charges", "senior_citizen", target]
corr = train[numeric_for_corr].corr()

display(corr.round(2))

fig, ax = plt.subplots(figsize=(7, 5.5))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=10)
ax.set_title("Numeric correlations: a first look, not a fitted model")
plt.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


### Discuss 1 — read the correlations

1. Which numeric variable has the strongest relationship with churn?
2. What can this table tell us?
3. What can it **not** tell us that logistic regression can?


In [ ]:
contract_rates = (
    train.groupby("contract", observed=True)[target]
    .agg(churn_rate="mean", customers="size")
    .sort_values("churn_rate")
)
display(contract_rates.style.format({"churn_rate": "{:.1%}"}))

ax = contract_rates["churn_rate"].plot(kind="barh", color="#2a63ad", figsize=(7, 3.8))
ax.set_xlabel("Observed churn rate")
ax.set_ylabel("Contract")
ax.set_title("A categorical relationship: churn rate by contract")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout()
plt.show()


# Mini Lab Part 1 — Build ordinary logistic regression slowly

**Goal:** see what logistic regression adds beyond correlation, then finish with a working baseline you can evaluate.

We will:

- compare one-input, two-input, and full-feature logistic models;
- interpret coefficients as changes in odds while other included inputs stay fixed;
- inspect how predicted probabilities change for actual customers;
- create one stratified train/validation split;
- evaluate all three models on the same validation rows;
- replace dependence on one split with stratified K-fold cross-validation.


## 1. Prevalence and the simple baseline

If the churn rate is low, a model can have high accuracy by predicting that everyone stays. That is why accuracy alone is not enough.


In [ ]:
target = "churned"
prevalence = train[target].mean()
always_stay_accuracy = 1 - prevalence

summary = pd.Series({
    "churn prevalence": prevalence,
    "always-stay accuracy": always_stay_accuracy,
})
display(summary.to_frame("value").style.format("{:.1%}"))

ax = summary.plot(kind="bar", color=["#2a63ad", "#b54740"], ylim=(0, 1), rot=0)
ax.set_ylabel("Share of customers")
ax.set_title("Class prevalence and the always-stay baseline")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1%}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()


### Discuss 2 — establish the baseline

1. What would accuracy report for the always-stay rule?
2. What does that number fail to tell us about customers who actually churn?


## 2. Train, validation, and test are different roles

- **Training rows** are used to learn preprocessing and coefficients.
- **Validation rows** estimate performance and help us compare choices.
- **Test rows** should remain untouched until the policy is finalized.

Stratification keeps approximately the same churn prevalence on both sides of this first split.


In [ ]:
X = train.drop(columns=[target, "customer_id"])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

pd.DataFrame({
    "rows": [len(y_train), len(y_valid)],
    "churn rate": [y_train.mean(), y_valid.mean()],
}, index=["train", "validation"])


## 3. Model A — one input

Start with `tenure_months`. Before fitting, predict the coefficient sign:

- positive means longer tenure raises the odds of churn;
- negative means longer tenure lowers the odds of churn.

The odds multiplier for one additional month is $e^{β}$.


In [ ]:
one_features = ["tenure_months"]

one_input_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE)),
])
one_input_model.fit(X_train[one_features], y_train)

beta_tenure = one_input_model.named_steps["model"].coef_[0, 0]
print(f"tenure coefficient: {beta_tenure:.3f}")
print(f"odds multiplier for one additional month: {np.exp(beta_tenure):.3f}")

tenure_grid = pd.DataFrame({"tenure_months": np.arange(0, 73)})
tenure_probability = one_input_model.predict_proba(tenure_grid)[:, 1]
plt.figure(figsize=(7, 4))
plt.plot(tenure_grid["tenure_months"], tenure_probability, color="#2a63ad", linewidth=3)
plt.xlabel("Tenure in months")
plt.ylabel("Predicted probability of churn")
plt.ylim(0, 1)
plt.title("Model A: one input produces a probability curve")
plt.grid(alpha=0.2)
plt.show()


### Discuss 3 — coefficient, odds, probability

1. Does the coefficient sign match the correlation sign?
2. Does the odds multiplier describe odds or probability?
3. Is the change in probability constant across the whole curve?


## 4. Model B — add a second input

Now add `monthly_charges`. Each coefficient answers a conditional question: how does one input relate to churn **while the other included input stays fixed**?


In [ ]:
two_features = ["tenure_months", "monthly_charges"]
two_input_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE)),
])
two_input_model.fit(X_train[two_features], y_train)

two_coef = pd.DataFrame({
    "coefficient": two_input_model.named_steps["model"].coef_.ravel(),
}, index=two_features)
two_coef["odds multiplier for one unit"] = np.exp(two_coef["coefficient"])
display(two_coef.round(3))

example_customers = pd.DataFrame({
    "tenure_months": [6, 6, 48, 48],
    "monthly_charges": [35, 90, 35, 90],
})
example_customers["predicted churn probability"] = two_input_model.predict_proba(example_customers)[:, 1]
display(example_customers.style.format({"predicted churn probability": "{:.1%}"}))


### Discuss 4 — what changed after adding a variable?

1. Compare customers with the same tenure and different monthly charges.
2. Compare customers with the same monthly charges and different tenure.
3. Why can a coefficient change after another variable enters the model?


## 5. Model C — use the full feature set safely

Scaling, imputation, and category encoding learn values from data. Keeping them inside the pipeline ensures that each fit learns only from its training rows.


In [ ]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

baseline_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE)),
])

baseline_model


## 6. Compare Models A, B, and C on the same validation customers

The comparison changes only the feature set. The validation rows and metrics stay fixed.


In [ ]:
baseline_model.fit(X_train, y_train)
valid_probability = baseline_model.predict_proba(X_valid)[:, 1]

one_probability = one_input_model.predict_proba(X_valid[one_features])[:, 1]
two_probability = two_input_model.predict_proba(X_valid[two_features])[:, 1]

def probability_metrics(actual, probability):
    prediction = (probability >= 0.50).astype(int)
    return {
        "accuracy @ 0.50": accuracy_score(actual, prediction),
        "log loss": log_loss(actual, probability),
        "ROC-AUC": roc_auc_score(actual, probability),
        "PR-AUC": average_precision_score(actual, probability),
    }

model_comparison = pd.DataFrame({
    "Model A: tenure only": probability_metrics(y_valid, one_probability),
    "Model B: tenure + charges": probability_metrics(y_valid, two_probability),
    "Model C: full feature set": probability_metrics(y_valid, valid_probability),
}).T
display(model_comparison.round(3))

metrics = pd.Series({
    "log loss (lower is better)": log_loss(y_valid, valid_probability),
    "ROC-AUC": roc_auc_score(y_valid, valid_probability),
    "PR-AUC": average_precision_score(y_valid, valid_probability),
})
display(metrics.to_frame("validation result").round(3))

threshold = 0.50
valid_prediction = (valid_probability >= threshold).astype(int)
cm = confusion_matrix(y_valid, valid_prediction)

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["Predict stay", "Predict churn"])
ax.set_yticks([0, 1], labels=["Actually stay", "Actually churn"])
ax.set_title(f"Confusion matrix at threshold {threshold:.2f}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()


### Discuss 5 — evaluate the baseline model

1. Which metric answers “Are the probabilities numerically useful?”
2. Which error in the confusion matrix represents a customer who churned but was not contacted?
3. Change `threshold` from `0.50` to `0.30` and rerun only the confusion-matrix cell. What moves?


## 7. Why cross-validation?

One validation split may be unusually easy or difficult. In five-fold cross-validation, each fold serves as validation once while the other four folds train the complete pipeline.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "neg_log_loss": "neg_log_loss",
}

cv_result = cross_validate(baseline_model, X, y, cv=cv, scoring=scoring)
fold_results = pd.DataFrame({
    "ROC-AUC": cv_result["test_roc_auc"],
    "PR-AUC": cv_result["test_pr_auc"],
    "log loss": -cv_result["test_neg_log_loss"],
}, index=[f"fold {i}" for i in range(1, 6)])

display(fold_results.round(3))
display(pd.DataFrame({"mean": fold_results.mean(), "std": fold_results.std()}).round(3))


### Part 1 checkpoint — explain the complete baseline

In two or three sentences, record:

- what correlation showed before fitting a model;
- what changed from Model A to Model B to Model C;
- why the five-fold mean and spread are more informative than one validation score.


# Mini Lab Part 2 — Ridge and Lasso

**Goal:** redo Model C with three penalty choices while keeping the feature set, split logic, preprocessing, and metrics fixed.

- The near-unpenalized model provides the baseline.
- **Ridge (L2)** shrinks correlated signals and usually keeps all coefficients.
- **Lasso (L1)** can set some coefficients exactly to zero.

`C` is the inverse of regularization strength in scikit-learn: a smaller `C` means stronger shrinkage.


In [ ]:
models = {
    "near-unpenalized": LogisticRegression(C=1e6, max_iter=2000, random_state=RANDOM_STATE),
    "Ridge (L2)": LogisticRegression(penalty="l2", C=0.20, max_iter=2000, random_state=RANDOM_STATE),
    "Lasso (L1)": LogisticRegression(penalty="l1", solver="liblinear", C=0.20, max_iter=2000, random_state=RANDOM_STATE),
}

comparison_rows = []
fitted_pipelines = {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    result = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    comparison_rows.append({
        "model": name,
        "ROC-AUC mean": result["test_roc_auc"].mean(),
        "ROC-AUC std": result["test_roc_auc"].std(),
        "PR-AUC mean": result["test_pr_auc"].mean(),
        "log loss mean": -result["test_neg_log_loss"].mean(),
    })
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

comparison = pd.DataFrame(comparison_rows).set_index("model")
display(comparison.round(3))

print("Read this as a redo of Model C: the inputs and evaluation stay fixed; only the penalty changes.")


## Inspect what happened to the coefficients

The chart compares the largest coefficients after preprocessing. A positive coefficient increases log-odds of churn; a negative coefficient decreases them, holding the other encoded inputs fixed.


In [ ]:
feature_names = fitted_pipelines["near-unpenalized"].named_steps["preprocess"].get_feature_names_out()
coef_frame = pd.DataFrame(index=feature_names)

for name, pipe in fitted_pipelines.items():
    coef_frame[name] = pipe.named_steps["model"].coef_.ravel()

top_features = coef_frame["near-unpenalized"].abs().nlargest(12).index
plot_data = coef_frame.loc[top_features].sort_values("near-unpenalized")
plot_data.plot(kind="barh", figsize=(9, 6), color=["#b54740", "#2a63ad", "#27835f"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient on the log-odds scale")
plt.title("How Ridge and Lasso change the fitted coefficients")
plt.tight_layout()
plt.show()

zero_counts = (coef_frame.abs() < 1e-10).sum().rename("coefficients equal to zero")
zero_counts.to_frame()


### Discuss 6 — compare the redo

1. Which model produces the most exact zeros?
2. Did regularization materially change mean validation performance?
3. If two models perform similarly, what practical reason might favor Ridge, Lasso, or the simpler baseline?

Do not claim that a zero Lasso coefficient proves a feature is unimportant. Correlated features can exchange which signal is retained.


## After class — continue Exercise 02

Keep the same validation design while you extend the work. Preserve the fold definition, pipeline, baseline metrics, and assumptions so later comparisons remain fair.

### 4. Tree-based comparison

Add a tree or Random Forest after Week 6.

### 5. Threshold economics

Plot expected campaign value by threshold and conduct sensitivity analysis after Week 5.

### 6. Final policy and submission


In [ ]:
# probabilities = final_model.predict_proba(test.drop(columns="customer_id"))[:, 1]
# threshold = ...
# submission = pd.DataFrame({
#     "customer_id": test["customer_id"],
#     "churn_probability": probabilities,
#     "contact_customer": (probabilities >= threshold).astype(int),
# })
# submission.to_csv("../submission.csv", index=False)


## Generative AI use disclosure

Complete the disclosure specified in `../../shared/ai_use_disclosure.md`. Record how AI was used, what you verified, and what decisions remain your own.
